In [6]:
"""
PIPELINE STAGE: High-Integrity Multi-Source Data Fusion (Energy-Spatial-Solar)
INPUTS: 
    - 11_energy_population_geo.xlsx (Master Longitudinal Panel)
    - 12_daylight.xlsx (Reference Photoperiod Dataset)
OUTPUT: 13_energy_population_geo_day.xlsx
LIBRARIES: pandas, os

1. RESEARCH CORE: RELATIONAL KEY ALIGNMENT & DATA PURIFICATION
    This stage serves as the final synthesis point, merging dynamic time-series 
    data (Energy/Population) with static environmental metadata (Daylight). 
    The primary goal is to ensure that every observation across the 2020-2024 
    timeline is accurately enriched with geospatial photoperiod values.

2. METHODOLOGICAL SPECIFICATIONS & ERROR HANDLING:
    - Normalization of Relational Keys: Enforces strict integer casting on 
      'plate' and 'month' columns using 'pd.to_numeric' with 'coerce' parameters. 
      This robustly handles latent string artifacts (e.g., month names instead of IDs) 
      by converting them to NaNs and subsequent removal.
    - Attribute Standardization: Renames headers to a lowercase unified standard 
      (plate, month, province) to ensure seamless join operations regardless of 
      original spreadsheet casing.
    - Join Architecture: Utilizes a 'Left Join' strategy on a composite key 
      [plate, month] to broadcast monthly solar characteristics across the 
      multi-year master dataset.

3. DATA INTEGRITY VERIFICATION:
    - Dynamic Row Audit: Compares row counts before and after the merge to detect 
      data loss.
    - Null-Value Check: Implements an automated scan for unmapped 'daylight' cells, 
      providing a diagnostic warning if geospatial or seasonal gaps exist in 
      the source files.

4. ACADEMIC SIGNIFICANCE:
    The resulting unified dataset represents a complete 'Feature Space'. It allows 
    for the precise modeling of energy consumption patterns as a function of 
    demographic density (Population), geographical location (Lat/Long), 
    and solar availability (Daylight), which is crucial for econometric rigor.
"""

import pandas as pd
import os

def final_veri_birlesimi_temiz():
    # 1. Dosya Yolları
   
    MASTER_PATH = os.path.join("..", "..", "processed_data", "steps","11_energy_population_geo.xlsx")
    DAYLIGHT_PATH =os.path.join("..", "..", "processed_data", "steps", "12_daylight.xlsx")
    FINAL_OUT = os.path.join("..", "..", "processed_data", "steps","13_energy_population_geo_day.xlsx")

    try:
        # 2. Dosyaları Yükle
        print("Dosyalar yükleniyor...")
        df_master = pd.read_excel(MASTER_PATH)
        df_daylight = pd.read_excel(DAYLIGHT_PATH)

   
        # 3. KRİTİK ADIM: Her iki tabloda da anahtarları zorla sayıya çevir
        # errors='coerce' sayesinde eğer 'December' gibi bir metin varsa orayı NaN (boş) yapar
        print("Sayısal dönüşüm ve temizlik yapılıyor...")
        
        for col in ['Plate', 'Month']:
            df_master[col] = pd.to_numeric(df_master[col], errors='coerce')
            df_daylight[col] = pd.to_numeric(df_daylight[col], errors='coerce')

        # Hatalı (sayıya dönüştürülemeyen) satırları veri setinden atalım
        df_master.dropna(subset=['Plate', 'Month'], inplace=True)
        df_daylight.dropna(subset=['Plate', 'Month'], inplace=True)

        # Sayıları tam sayı (integer) yapalım ki join işlemi %100 eşleşsin

        if 'Daylight' in df_daylight.columns:
            df_daylight['Daylight'] = df_daylight['Daylight'].astype(str).str.replace(',', '.', regex=False)
            df_daylight['Daylight'] = pd.to_numeric(df_daylight['Daylight'], errors='coerce').fillna(0.0)
            
        df_master[['Plate', 'Month']] = df_master[['Plate', 'Month']].astype(int)
        df_daylight[['Plate', 'Month']] = df_daylight[['Plate', 'Month']].astype(int)

        

        # 4. Birleştirme (Left Join)
        # Sadece daylight sütununu master tabloya ekliyoruz
        print("Birleştirme işlemi uygulanıyor...")
        final_df = pd.merge(
            df_master, 
            df_daylight[['Plate', 'Month', 'Daylight']], 
            on=['Plate', 'Month'], 
            how='left'
        )

        # 5. Kaydet
        final_df.to_excel(FINAL_OUT, index=False)
        
        print("\n" + "="*50)
        print("İŞLEM TAMAMLANDI")
        print(f"Master Satır Sayısı: {len(df_master)}")
        print(f"Final Satır Sayısı: {len(final_df)}")
        print(f"Dosya: {FINAL_OUT}")
        print("="*50)
        
        # Kontrol: Daylight sütununda boş (eşleşmemiş) veri var mı?
        missing = final_df['Daylight'].isnull().sum()
        if missing > 0:
            print(f"UYARI: {missing} adet satırda daylight verisi eşleşmedi! Lütfen plaka/ay numaralarını kontrol edin.")
        else:
            print("KONTROL: Tüm satırlar başarıyla eşleşti.")

    except Exception as e:
        print(f"HATA: {e}")

if __name__ == "__main__":
    final_veri_birlesimi_temiz()

Dosyalar yükleniyor...
Sayısal dönüşüm ve temizlik yapılıyor...
Birleştirme işlemi uygulanıyor...

İŞLEM TAMAMLANDI
Master Satır Sayısı: 4860
Final Satır Sayısı: 4860
Dosya: ..\..\processed_data\steps\13_energy_population_geo_day.xlsx
KONTROL: Tüm satırlar başarıyla eşleşti.
